# [9-4강] 저장불러오기와 학습 재개 - 실습

In [2]:
import random
import json
import math
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset, random_split

# 실습 결과가 매번 비슷하게 나오도록 seed를 고정합니다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


## 필수 1 1: checkpoint 저장 함수 작성하기

### 문제 설명
모델과 optimizer 상태를 파일에 저장하는 함수를 만듭니다.

In [3]:
def save_checkpoint(path, model, optimizer, epoch):
    # TODO: 저장할 checkpoint를 구성하세요.
    ckpt = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict()
    }
    # TODO: torch.save로 저장하세요.
    torch.save(ckpt, path)
    return Path(path).exists()

model = nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
path = 'resume_ckpt.pt'
print('saved:', save_checkpoint(path, model, optimizer, 1))
Path(path).unlink(missing_ok=True)

saved: True


### 해설 및 실행 결과 해석
저장 함수로 분리하면 학습 중 best model이나 특정 epoch checkpoint를 반복해서 남기기 쉽습니다.

## 필수 2 2: checkpoint 로드 함수 작성하기

### 문제 설명
저장된 checkpoint를 새 모델과 optimizer에 불러옵니다.

In [4]:
path = 'load_ckpt.pt'
model = nn.Linear(2, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
torch.save({'epoch': 5, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict()}, path)

def load_checkpoint(path, model, optimizer):
  ckpt = torch.load(path, map_location='cpu')
  model.load_state_dict(ckpt['model_state'])
  optimizer.load_state_dict(ckpt['optimizer_state'])
  return ckpt['epoch']
    # TODO: checkpoint를 읽고 model/optimizer를 복원하세요.

new_model = nn.Linear(2, 1)
new_optimizer = torch.optim.Adam(new_model.parameters(), lr=0.01)
print('loaded_epoch:', load_checkpoint(path, new_model, new_optimizer))
Path(path).unlink(missing_ok=True)

loaded_epoch: 5


In [5]:
####
path = 'load_ckpt.pt'
model = nn.Linear(2, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
torch.save({'epoch': 5, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict()}, path)

def load_checkpoint(path, model, optimizer):
    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    return ckpt['epoch']

new_model = nn.Linear(2, 1)
new_optimizer = torch.optim.Adam(new_model.parameters(), lr=0.01)
print('loaded_epoch:', load_checkpoint(path, new_model, new_optimizer))
Path(path).unlink(missing_ok=True)

loaded_epoch: 5


### 해설 및 실행 결과 해석
epoch까지 함께 복원해야 어느 시점부터 이어서 학습해야 하는지 알 수 있습니다. optimizer state는 Adam처럼 내부 상태가 있는 optimizer에서 특히 중요합니다.